# Library

In [1]:
import fitz
import os
from openai import OpenAI
import json
import requests

# OCR Extraction (PDF TEXTABLE)

In [2]:
def extraction(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text = text + page.get_text()
    return text


# OCR Extraction (OCR API KEY)

In [ ]:
def ocr_space(file_path,api_key = "maksed"):
    url = "http://api.ocr.space/parse/image"

    with open(file_path,"rb") as f:
        response = requests.post(
            url , 
            files = {"file":f},
            data ={
                "apikey":api_key,
                "language":"eng"
            }
        )
    
    result = response.json()
    text_data = result["ParsedResults"][0]["ParsedText"]

    return text_data


# Length of Text


In [29]:
def text_len(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text = text + page.get_text()
    return len(text)


# LLM API KEY

In [ ]:
os.environ['OPENAI_API_KEY'] = "masked"
client = OpenAI(
    base_url="https://api.groq.com/openai/v1"
)

# Full function

In [4]:
def final_CV_details(text):
    prompt = f"""You are an expert HR information extraction system.

Extract structured information from the CV text.

Rules for extracting profile details:
- Return summary of profile in the CV in profile_summary section.

Rules for extracting skills:
- Return all technical and soft skills in the CV in profile_summary and soft_skills section.

Rules for education:
- Extract O/L, A/L, degree, and certificate information if available.
- Set "highest_qualification" to the highest completed qualification (ol, al, diploma, hnd, degree, masters, phd).

Rules for experiences:
- Calculate duration_months for each job: (end_year*12 + end_month) - (start_year*12 + start_month).
- If end_date="Present", use month of today.
- total_experience_years = sum(all duration_months) / 12, round to 1 decimal

Rules for project:
- Return summary of each projects mentioned in the CV in project_details section.
- Must Return technical or related keywords mentioned in the CV in keywords section.

Common rules for every section:
- Return ONLY valid JSON.
- Do not add or remove fields.
- Use null for missing values.

JSON STRUCTURE:
{{
  "candidate_name": "",
  "candidate_contact": {{
    "email": "",
    "phone": "",
    "location": ""
  }},

  "profile_summary": "",

  "technical_skills": [],
  "soft_skills": [],

  "education": {{
    "highest_qualification": null,

    "ol": {{
      "year": null,
      "subjects": [
        {{
          "subject": "",
          "grade": ""
        }}
      ]
    }},

    "al": {{
      "year": null,
      "stream": null,
      "subjects": [
        {{
          "subject": "",
          "grade": ""
        }}
      ],
      "z_score": null
    }},

    "degrees": [
      {{
        "degree_name": "",
        "university": "",
        "gpa": null
      }}
    ],

    "certificates": [
      {{
        "name": ""
      }}
    ]
  }},

  "experience": {{
    "total_experience_years": null,
    "jobs": [
      {{
        "job_title": "",
        "company": "",
        "duration": ""
      }}
    ]
  }},

  "projects": [
    {{
      "project_name": "",
      "project_details": "",
      "keywords": []
    }}
  ]
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 1500,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

# Data set

In [5]:
if __name__ == "__main__":
    data = extraction("Associate Data Scientist Induwara Dilshan.pdf")
    datafile = final_CV_details(data)
    with open("one_cv_extraction.json","w", encoding = "utf-8") as f:
        json.dump(datafile,f,indent= 4)

'CONTACT\nSKILLS & COMPETENCIES\n076 4390379\ninduwaradilshan7@gmail.com\nKurunegala\nGitHub\nINDUWARA DILSHAN \n Data Science Engineer\nEDUCATION\nPROFILE\nAssociate Data Scientist/ AI Engineer with hands-on internship\nexperience in machine learning and NLP. Skilled in building\npredictive models, time series forecasting, and dashboards with\ndata analysis using Python and modern AI tools. Experienced in\nworking with real-world financial data during an internship,\napplying analytical techniques to generate insights.\nBachelor of Science in Statistics (Special)\nDepartment of Statistics and Computer Science, Faculty of Science, \nUniversity of Peradeniya.\nGPA : 3.61\nRelevant Coursework :\nProgramming\nPython, R, C\nMachine Learning\nML model\nDeep Learning\nTransformer\nLLMs\nTime Series Prediction\nMLOps\nCI/ CD pipeline\nA/B Testing\nFastAPI Framework\nOther AI \nRAG\nEmbedding\nFine Tuning LLM\nOpenAI API, Llama API\nLlama Index/ Model\nInduwara_Dilshan\nEXPERIENCE\nCentral Ban

# END 

# TESTING PARTS

### Experiences

In [7]:
def experience(text):
    prompt = f"""You are an expert HR information extraction system.

Extract structured information from the CV text.

RULES:
- Return ONLY valid JSON
- Calculate duration_months for each job: (end_year*12 + end_month) - (start_year*12 + start_month)
- If end_date="Present", use June 15, 2026 as end_date
- total_experience_years = sum(all duration_months) / 12, round to 1 decimal

JSON STRUCTURE:
{{
  "experience": [
    {{
    "total_experience_years": null,
    "jobs": [
      {{
        "job_title": "",
        "company": "",
        "Duration: null
      }}
    ]
    }}
]
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 1500,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

### Education

In [8]:
def education(text):
    prompt = f"""You are an expert HR information extraction system.

Extract structured information from the CV text.

Rules:
- Return ONLY valid JSON.
- Do not add or remove fields.
- Use null for missing values.
- Extract O/L, A/L, degree, and certificate information if available.
- Set "highest_qualification" to the highest completed qualification (ol, al, diploma, hnd, degree, masters, phd).


JSON STRUCTURE:
{{
  "education": {{
    "highest_qualification": null,

    "ol": {{
      "year": null,
      "subjects": [
        {{
          "subject": "",
          "grade": ""
        }}
      ]
    }},

    "al": {{
      "year": null,
      "stream": null,
      "subjects": [
        {{
          "subject": "",
          "grade": ""
        }}
      ],
      "z_score": null
    }},

    "degrees": [
      {{
        "degree_name": "",
        "university": "",
        "gpa": null
      }}
    ],

    "certificates": [
      {{
        "name": ""
      }}
    ]
  }}
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 150,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

### Project details

In [9]:
def project(text):
    prompt = f"""You are an expert HR information extraction system.

Extract structured information from the CV text.

Rules:
- Return ONLY valid JSON.
- Do not add or remove fields.
- Return summary of each projects mentioned in the CV in project_details section.
- Must Return technical or related keywords mentioned in the CV in keywords section.


JSON STRUCTURE:
{{
  "projects": [
    {{
      "project_name": "",
      "project_details": "",
      "keywords": []
    }}
  ]
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 800,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

### profile structure

In [10]:
def profile(text):
    prompt = f"""You are an expert HR information extraction system.

Extract structured information from the CV text.

Rules:
- Return ONLY valid JSON.
- Do not add or remove fields.
- Return summary of profile in the CV in profile_summary section.


JSON STRUCTURE:
{{
  "candidate_name": "",
  "candidate_contact": {{
    "email": "",
    "phone": "",
    "location": ""
  }},

  "profile_summary": ""
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 800,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

### Skill

In [ ]:
def skills(text):
    prompt = f"""You are an expert HR information extraction system.

Extract structured information from the CV text.

Rules:
- Return ONLY valid JSON.
- Do not add or remove fields.
- Return all technical and soft skills in the CV in profile_summary and soft_skills section.


JSON STRUCTURE:
{{
    "Technical_skills": "",
    "soft_skills":""
}}

TEXT:
{text}
"""
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{
            "role":"user",
            "content" : prompt
        }],
        temperature=0.0,
        max_tokens = 100,
        response_format={"type": "json_object"}
    )

    output = response.choices[0].message.content.strip()

    try:
        data =  json.loads(output)
    except json.JSONDecodeError:
        print("JSON Error!!!")
        data = {}

    return data

# Final Function

In [14]:
def collection(text_data):
    main_file = {}
    main_file.update(profile(text_data))
    main_file.update(education(text_data))
    main_file.update(skills(text_data))
    main_file.update(project(text_data))
    main_file.update(experience(text_data))

    with open("one_cv_extraction_collection.json","w",encoding="utf-8") as f:
        json.dump(main_file,f,indent=4)

    return main_file

In [ ]:
# if __name__ == "__main__":
#     number = text_len("Associate Data Scientist Induwara Dilshan.pdf")
#     if number != 0:
#         text_data = extraction("Associate Data Scientist Induwara Dilshan.pdf")
#         result = collection(text_data)
#         print("Successfully!!!")
#     else:
#         text_data = ocr_space("")

Successfully!!!
